## Agent Engine Evaluation

In [ ]:
%pip uninstall -y vertexai
%pip install google-cloud-aiplatform google-cloud-trace --upgrade
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [1]:
import google.auth
credentials, PROJECT_ID = google.auth.default()
PROJECT_ID

'sandbox-373102'

In [1]:
LOCATION = "us-central1"
import json
with open("../deployment_metadata.json", 'r', encoding='utf-8') as file:
    REASONING_ENGINE_ID = json.load(file)['remote_agent_engine_id']
REASONING_ENGINE_ID

'projects/1045259343465/locations/us-central1/reasoningEngines/618986013775101952'

In [ ]:
import time
import vertexai
from vertexai import Client
from google.genai import types as genai_types

STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-engine"
GCS_DEST = f"{STAGING_BUCKET}/output-path"
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

client = Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=genai_types.HttpOptions(api_version="v1beta1"),
)

!gcloud storage buckets create "{STAGING_BUCKET}"

Creating gs://sandbox-373102-agent-engine/...
ERROR: (gcloud.storage.buckets.create) HTTPError 409: The requested bucket name is not available. The bucket namespace is shared by all users of the system. Please select a different name and try again.


In [4]:
#Due to Jupyter issue, isolate rendering function from sdk
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import json
from vertexai._genai import types
import vertexai._genai._evals_visualization as sdk
from typing import Optional
from pydantic import errors
import base64
def display_evaluation_dataset(eval_dataset_obj: types.EvaluationDataset) -> None:
    """Displays an evaluation dataset in an IPython environment."""
    from IPython import display

    processed_rows = []
    df = eval_dataset_obj.eval_dataset_df

    for _, row in df.iterrows():
        processed_row = {}
        for col_name, cell_value in row.items():
            if col_name in ["prompt", "request", "response"]:
                processed_row[col_name] = sdk._extract_text_and_raw_json(cell_value)
            elif col_name == "rubric_groups":
                # Special handling for rubric_groups to keep it as a dict
                if isinstance(cell_value, dict):
                    processed_row[col_name] = {
                        k: [
                            (
                                v_item.model_dump(mode="json")
                                if hasattr(v_item, "model_dump")
                                else v_item
                            )
                            for v_item in v
                        ]
                        for k, v in cell_value.items()
                    }
                else:
                    processed_row[col_name] = cell_value
            else:
                if isinstance(cell_value, (dict, list)):
                    processed_row[col_name] = json.dumps(
                        cell_value, ensure_ascii=False, default=sdk._pydantic_serializer
                    )
                else:
                    processed_row[col_name] = cell_value
        processed_rows.append(processed_row)

    dataframe_json_string = json.dumps(processed_rows, ensure_ascii=False, default=str)
    html_content = sdk._get_inference_html(dataframe_json_string)
    b64_html = base64.b64encode(html_content.encode('utf-8')).decode('utf-8')
    data_uri = f"data:text/html;charset=utf-8;base64,{b64_html}"
    display.display(display.IFrame(data_uri, width="100%", height="800px"))

def display_evaluation_result(eval_result_obj: types.EvaluationResult, candidate_names: Optional[list[str]] = None) -> None:
    """Displays evaluation result in an IPython environment."""
    from IPython import display

    try:
        result_dump = eval_result_obj.model_dump(
            mode="json", exclude_none=True, exclude={"evaluation_dataset"}
        )
    except errors.PydanticSerializationError as e:
        print(
            "Serialization Error: %s\nCould not display the evaluation "
            "result due to a data serialization issue. Please check the "
            "content of the EvaluationResult object.",
            e,
        )
        return
    except Exception as e:
        print("Failed to serialize EvaluationResult: %s", e, exc_info=True)
        raise

    input_dataset_list = eval_result_obj.evaluation_dataset
    is_comparison = input_dataset_list and len(input_dataset_list) > 1

    metadata_payload = result_dump.get("metadata", {})
    metadata_payload["candidate_names"] = candidate_names or metadata_payload.get(
        "candidate_names"
    )

    if is_comparison and input_dataset_list:
        if input_dataset_list[0]:
            metadata_payload["dataset"] = sdk._extract_dataset_rows(input_dataset_list[0])

        if "eval_case_results" in result_dump:
            for case_res in result_dump["eval_case_results"]:
                for resp_idx, cand_res in enumerate(
                    case_res.get("response_candidate_results", [])
                ):
                    if (
                        input_dataset_list is not None
                        and resp_idx < len(input_dataset_list)
                        and input_dataset_list[resp_idx]
                    ):
                        rows = sdk._extract_dataset_rows(input_dataset_list[resp_idx])
                        case_idx = case_res.get("eval_case_index")
                        if case_idx is not None and case_idx < len(rows):
                            original_case = rows[case_idx]
                            cand_res["display_text"] = original_case[
                                "response_display_text"
                            ]
                            cand_res["raw_json"] = original_case["response_raw_json"]

        win_rates = eval_result_obj.win_rates if eval_result_obj.win_rates else {}
        if "summary_metrics" in result_dump:
            for summary in result_dump["summary_metrics"]:
                if summary.get("metric_name") in win_rates:
                    summary.update(win_rates[summary["metric_name"]])

        result_dump["metadata"] = metadata_payload
        html_content = sdk._get_comparison_html(json.dumps(result_dump))
    else:
        single_dataset = input_dataset_list[0] if input_dataset_list else None
        processed_rows = []
        if single_dataset is not None:
            processed_rows = sdk._extract_dataset_rows(single_dataset)
            metadata_payload["dataset"] = processed_rows

            if "eval_case_results" in result_dump and processed_rows:
                for case_res in result_dump["eval_case_results"]:
                    case_idx = case_res.get("eval_case_index")
                    if (
                        case_idx is not None
                        and case_idx < len(processed_rows)
                        and case_res.get("response_candidate_results")
                    ):
                        original_case = processed_rows[case_idx]
                        cand_res = case_res["response_candidate_results"][0]
                        cand_res["display_text"] = original_case[
                            "response_display_text"
                        ]
                        cand_res["raw_json"] = original_case["response_raw_json"]

        result_dump["metadata"] = metadata_payload
        html_content = sdk._get_evaluation_html(json.dumps(result_dump))

    b64_html = base64.b64encode(html_content.encode('utf-8')).decode('utf-8')
    data_uri = f"data:text/html;charset=utf-8;base64,{b64_html}"
    display.display(display.IFrame(data_uri, width="100%", height="800px"))

Evaluate agent that locally exist in code

Evaluate deployed agent

In [5]:
#Prepare sample data
import pandas as pd
from vertexai._genai import types

session_inputs = types.evals.SessionInput(
    user_id="Test user",
    state={},
)
agent_prompts = [
    "치킨 커리 레시피가 궁금해요",
    "1주일 채식주의자를 위한 식단을 알려주세요",
]
agent_dataset = pd.DataFrame({
    "prompt": agent_prompts,
    "session_inputs": [session_inputs] * len(agent_prompts),
})
agent_dataset

,prompt,session_inputs
0,치킨 커리 레시피가 궁금해요,user_id='Test user' state={} app_name=None
1,1주일 채식주의자를 위한 식단을 알려주세요,user_id='Test user' state={} app_name=None


In [7]:
from google.genai import types as genai_types
from vertexai._genai import types
httpOptions = genai_types.HttpOptions(
    retry_options=genai_types.HttpRetryOptions(
        attempts=5,           # 최대 재시도 횟수
        initial_delay=1.0,    # 첫 대기 시간
        http_status_codes=[429, 500, 502, 503, 504] # 재시도 대상 에러 코드
    )
)

#Get response
eval_dataset = client.evals.run_inference(
    agent=REASONING_ENGINE_ID,
    src=agent_dataset,
    config=types.EvalRunInferenceConfig(
        generate_content_config=genai_types.GenerateContentConfig(
            httpOptions=httpOptions)
            )
)

display_evaluation_dataset(eval_dataset)

Agent Run: 100%|██████████| 2/2 [00:25<00:00, 12.53s/it]


In [8]:
evaluation_run = client.evals.create_evaluation_run(
    dataset=eval_dataset,
    agent=REASONING_ENGINE_ID,
    metrics=[
        types.RubricMetric.FINAL_RESPONSE_QUALITY,
        types.RubricMetric.TOOL_USE_QUALITY,
        types.RubricMetric.HALLUCINATION,
        types.RubricMetric.SAFETY,
    ],
    dest=GCS_DEST,
    config=types.CreateEvaluationRunConfig(http_options=httpOptions)
)

C:\Users\jeehyeok\AppData\Local\Temp\ipykernel_12644\2145927774.py:1: ExperimentalWarning: The Vertex SDK GenAI evals.create_evaluation_run module is experimental, and may change in future versions.
  evaluation_run = client.evals.create_evaluation_run(
c:\Users\jeehyeok\Documents\agent_engine\agent-engine-lab\.venv\Lib\site-packages\vertexai\_genai\_evals_common.py:2964: ExperimentalWarning: The Vertex SDK GenAI evals.create_evaluation_item module is experimental, and may change in future versions.
  eval_item = evals_module.create_evaluation_item(
c:\Users\jeehyeok\Documents\agent_engine\agent-engine-lab\.venv\Lib\site-packages\vertexai\_genai\_evals_common.py:2971: ExperimentalWarning: The Vertex SDK GenAI evals.create_evaluation_set module is experimental, and may change in future versions.
  evaluation_set = evals_module.create_evaluation_set(


In [9]:
while evaluation_run.state not in {"SUCCEEDED", "FAILED", "CANCELLED"}:
    evaluation_run.show()
    evaluation_run = client.evals.get_evaluation_run(name=evaluation_run.name)
    time.sleep(10)

evaluation_run = client.evals.get_evaluation_run(
    name=evaluation_run.name, include_evaluation_items=True
)

# Display the Evaluation Run status and results
display_evaluation_result(evaluation_run.evaluation_item_results)

C:\Users\jeehyeok\AppData\Local\Temp\ipykernel_12644\871509031.py:3: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_run module is experimental, and may change in future versions.
  evaluation_run = client.evals.get_evaluation_run(name=evaluation_run.name)


c:\Users\jeehyeok\Documents\agent_engine\agent-engine-lab\.venv\Lib\site-packages\vertexai\_genai\_evals_common.py:2758: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_set method is experimental, and may change in future versions.
  eval_set = evals_module.get_evaluation_set(
c:\Users\jeehyeok\Documents\agent_engine\agent-engine-lab\.venv\Lib\site-packages\vertexai\_genai\_evals_common.py:2765: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_item method is experimental, and may change in future versions.
  evals_module.get_evaluation_item(name=item_name)
